# Work Hour Prediction Model

This model estimates how many work hours an asphalt job will take.

It is trained on 20 example projects based on my experience on past asphalt jobs. Its answers are estimates for planning, not guaranteed numbers.

In [1]:
import random
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error

## 1. Make the data

This creates 20 example projects (5 of each type) based on my experience on similar jobs.

- Every job has setup time, and bigger jobs get bigger crews
- Paving time goes up with size and thickness, and down with more workers
- Fair pavement adds about 15% more time, poor pavement about 35%
- Some variation is added, like on real jobs

`random.seed(42)` creates the same 20 projects every time. When job records are collected through the website, they replace `past_projects.csv`.

In [2]:
random.seed(42)

project_types = ["New driveway", "Driveway replacement", "Parking lot", "Resurfacing"] * 5
random.shuffle(project_types)

setup_hours = {"New driveway": 6, "Driveway replacement": 8, "Parking lot": 10, "Resurfacing": 4}
condition_extra = {"Good": 1.0, "Fair": 1.15, "Poor": 1.35}

projects = []

for project_type in project_types:
    if project_type == "Parking lot":
        square_footage = random.randint(3000, 20000)
        thickness = random.choice([3, 3.5, 4])
        crew_size = random.randint(5, 8)
    elif project_type == "Resurfacing":
        square_footage = random.randint(500, 10000)
        thickness = random.choice([1.5, 2])
        crew_size = random.randint(3, 6)
    else:
        square_footage = random.randint(400, 1500)
        thickness = random.choice([2, 2.5, 3])
        crew_size = random.randint(3, 5)

    condition = random.choice(["Good", "Fair", "Poor"])

    paving_hours = square_footage * thickness / (150 * crew_size)
    work_hours = (setup_hours[project_type] + paving_hours) * condition_extra[condition]
    work_hours = round(work_hours * random.uniform(0.9, 1.1), 1)

    projects.append({
        "project_type": project_type,
        "square_footage": square_footage,
        "thickness": thickness,
        "crew_size": crew_size,
        "condition": condition,
        "work_hours": work_hours,
    })

data = pd.DataFrame(projects)
data.to_csv("past_projects.csv", index=False)
data

,project_type,square_footage,thickness,crew_size,condition,work_hours
0,Resurfacing,9695,1.5,6,Good,20.0
1,Driveway replacement,969,2.0,3,Poor,16.4
2,Parking lot,12105,3.0,6,Fair,53.3
3,New driveway,1178,2.0,4,Fair,11.7
4,Driveway replacement,488,3.0,4,Poor,13.0
5,Driveway replacement,1175,2.0,5,Fair,13.6
6,Resurfacing,6425,1.5,3,Good,26.2
7,Parking lot,12482,3.0,6,Good,50.4
8,Parking lot,17857,4.0,7,Good,76.0
9,New driveway,829,3.0,4,Poor,14.9


## 2. Turn words into numbers

The model only understands numbers, so each project type and condition gets a number.

`X` is what the model looks at. `y` is the answer it tries to guess.

In [3]:
type_numbers = {"New driveway": 1, "Driveway replacement": 2, "Parking lot": 3, "Resurfacing": 4}
condition_numbers = {"Good": 1, "Fair": 2, "Poor": 3}

data["type_number"] = data["project_type"].map(type_numbers)
data["condition_number"] = data["condition"].map(condition_numbers)

X = data[["square_footage", "thickness", "crew_size", "type_number", "condition_number"]]
y = data["work_hours"]

## 3. Train the model

A Random Forest makes 100 decision trees and averages their guesses.

We use a **Regressor** because we are guessing a number (hours).

In [4]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)
print("Model trained on", len(X), "projects")

Model trained on 20 projects


## 4. Test the model

We split the 20 projects into 5 groups. Each group gets tested by a model that learned from the other 4 groups.

**Mean Absolute Error** = how many hours the guesses are off, on average. Smaller is better.

We compare it to just guessing the average every time.

In [5]:
predictions = cross_val_predict(RandomForestRegressor(n_estimators=100, random_state=42), X, y, cv=5)

model_error = mean_absolute_error(y, predictions)
average_guess_error = mean_absolute_error(y, [y.mean()] * len(y))

print("Just guessing the average is off by", round(average_guess_error, 1), "hours")
print("Our model is off by", round(model_error, 1), "hours")

data["predicted_hours"] = predictions.round(1)
data[["project_type", "square_footage", "work_hours", "predicted_hours"]]

Just guessing the average is off by 15.9 hours
Our model is off by 5.9 hours


,project_type,square_footage,work_hours,predicted_hours
0,Resurfacing,9695,20.0,35.1
1,Driveway replacement,969,16.4,11.9
2,Parking lot,12105,53.3,45.7
3,New driveway,1178,11.7,10.9
4,Driveway replacement,488,13.0,13.2
5,Driveway replacement,1175,13.6,10.3
6,Resurfacing,6425,26.2,21.1
7,Parking lot,12482,50.4,52.5
8,Parking lot,17857,76.0,50.0
9,New driveway,829,14.9,11.5


## 5. Save the model to a file

In [6]:
joblib.dump(model, "work_hours_model.joblib")
print("Model saved to work_hours_model.joblib")

Model saved to work_hours_model.joblib


## 6. Load the model from the file

In [7]:
loaded_model = joblib.load("work_hours_model.joblib")

## 7. Make a prediction

New project: 800 sq ft driveway replacement (type 2), 2.5 inches thick, crew of 4, poor condition (3).

In [8]:
new_project = pd.DataFrame([{
    "square_footage": 800,
    "thickness": 2.5,
    "crew_size": 4,
    "type_number": 2,
    "condition_number": 3,
}])

guess = loaded_model.predict(new_project)[0]
print("Estimated work hours:", round(guess, 1))
print("This is an estimate, not a guaranteed number.")

Estimated work hours: 13.6
This is an estimate, not a guaranteed number.


## 8. Try your own project

Run this cell and type in each detail, pressing Enter after each one.

Run it again to try another project.

In [ ]:
print("Project types: 1 = New driveway, 2 = Driveway replacement, 3 = Parking lot, 4 = Resurfacing")
print("Conditions: 1 = Good, 2 = Fair, 3 = Poor")
print()

typed_project = pd.DataFrame([{
    "square_footage": int(input("Square footage: ")),
    "thickness": float(input("Thickness in inches: ")),
    "crew_size": int(input("Crew size: ")),
    "type_number": int(input("Project type number (1-4): ")),
    "condition_number": int(input("Condition number (1-3): ")),
}])

typed_guess = loaded_model.predict(typed_project)[0]
print()
print("Estimated work hours:", round(typed_guess, 1))
print("This is an estimate, not a guaranteed number.")